# 🌿 Deteksi Penyakit Daun Teh: Faster R-CNN + Coordinate Attention (CA) + Edge Contour
Notebook terpadu (**End-to-End & Efisien**) untuk Skripsi (Komparasi Fase 3 — Two-Stage Detector):
1. **Google Drive Integration**: Otomatis mendeteksi dan mengekstrak `tea_yolo.zip` dari Google Drive.
2. **Arsitektur Faster R-CNN + CA**: Menggunakan `ResNet-50-FPN` + modul `CoordAtt` (Hou et al., CVPR 2021) pada seluruh level piramida fitur FPN (`P2, P3, P4, P5, P6`) sebelum `RPN` dan `RoIAlign`.
3. **100% COCO Weight Preservation + AMP**: Mempertahankan seluruh bobot pretrained COCO `FasterRCNN_ResNet50_FPN_Weights.DEFAULT` dan menggunakan *Mixed Precision (AMP)* agar pelatihan di GPU Colab berjalan cepat.
4. **Evaluasi Lengkap**: Evaluasi Held-Out Test Set (529 citra, 640×640), Confusion Matrix, PR-Curve, dan Kurva Konvergensi HD.
5. **In-Situ Bush Detector (v2.7) + Fine-Tuning Lapangan**: Multi-deteksi daun fokus depan, *Background Blur (Bokeh)*, eliminasi ranting/pakis/tanah, *white-twig healing*, 2-View TTA, serta alur Pra-Label & Fine-Tuning Data Lapangan.

### 1. Inisialisasi Environment, Google Drive & Dataset

In [ ]:
import os
import sys
import glob
import time
import math
import copy
import json
import random
import shutil
import zipfile
from pathlib import Path

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install library pendukung (PyTorch & Torchvision sudah terpasang di Colab)
!pip install -q opencv-python-headless matplotlib pandas

import numpy as np
import torch
import torchvision

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] PyTorch: {torch.__version__} | Torchvision: {torchvision.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[INFO] GPU Aktif: {torch.cuda.get_device_name(0)}")

# 3. Cari dan Ekstrak Dataset tea_yolo.zip dari Google Drive
# (Faster R-CNN tidak memakai data.yaml; hanya memakai citra & anotasi kotak yang sama agar komparasi adil)
DATA_DIR = "/content/tea_yolo"

if not os.path.exists(f"{DATA_DIR}/images/train"):
    zip_candidates = (
        glob.glob('/content/drive/MyDrive/SKRIPSI/tea_yolo.zip') +
        glob.glob('/content/drive/MyDrive/**/tea_yolo.zip', recursive=True)
    )
    if not zip_candidates:
        raise FileNotFoundError("Berkas 'tea_yolo.zip' tidak ditemukan di Google Drive!")

    zip_path = zip_candidates[0]
    print(f"[INFO] Mengekstrak dataset dari: {zip_path} ...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/')
    print(f"[OK] Dataset berhasil diekstrak ke: {DATA_DIR}")
else:
    print(f"[OK] Dataset sudah tersedia di: {DATA_DIR}")

for sp in ['train', 'val', 'test']:
    n_imgs = len(glob.glob(f"{DATA_DIR}/images/{sp}/*.*"))
    print(f"  - Split {sp:<5}: {n_imgs} citra")


### 2. Definisi Arsitektur Faster R-CNN-ResNet50-FPN + Coordinate Attention (CA)

In [ ]:
from collections import OrderedDict
import torch.nn as nn
import torchvision.models.detection as d_models
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Konstanta Kelas & Warna Resmi (Sama persis dengan Pipeline YOLO26n + CA)
CLASS_NAMES = [
    "Tea algal leaf spot", "Brown Blight", "Gray Blight",
    "Helopeltis", "Red Spider", "Green Mirid Bug", "Healty Leaf"
]
CLASS_COLORS = [
    (0, 140, 255),   # 0: Tea algal leaf spot - Orange
    (0, 0, 255),     # 1: Brown Blight        - Red
    (0, 215, 255),   # 2: Gray Blight         - Gold/Yellow
    (255, 0, 255),   # 3: Helopeltis          - Magenta
    (50, 50, 255),   # 4: Red Spider          - Coral Red
    (0, 255, 0),     # 5: Green Mirid Bug     - Green
    (255, 255, 0),   # 6: Healty Leaf         - Cyan
]

class CoordAtt(nn.Module):
    """Coordinate Attention Block (Hou et al., CVPR 2021)."""
    def __init__(self, inp: int = 256, oup: int = None, reduction: int = 16):
        super().__init__()
        oup = oup or inp
        mip = max(8, inp // reduction)
        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish()
        self.conv_h = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0, bias=True)
        self.conv_w = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        B, C, H, W = x.shape
        x_h = torch.mean(x, dim=3, keepdim=True)
        x_w = torch.mean(x, dim=2, keepdim=True).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))
        x_h_enc, x_w_enc = torch.split(y, [H, W], dim=2)
        x_w_enc = x_w_enc.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h_enc).sigmoid()
        a_w = self.conv_w(x_w_enc).sigmoid()
        return identity * a_h * a_w

class BackboneFPNWithCA(nn.Module):
    """Menyisipkan Coordinate Attention pada setiap skala fitur FPN (P2, P3, P4, P5, pool)."""
    def __init__(self, base_backbone: nn.Module, reduction: int = 16):
        super().__init__()
        self.body = base_backbone.body
        self.fpn = base_backbone.fpn
        self.out_channels = base_backbone.out_channels
        self.ca_blocks = nn.ModuleDict({
            k: CoordAtt(self.out_channels, self.out_channels, reduction=reduction)
            for k in ['0', '1', '2', '3', 'pool']
        })

    def forward(self, x: torch.Tensor):
        feats = self.fpn(self.body(x))
        out = OrderedDict()
        for k, v in feats.items():
            out[k] = self.ca_blocks[k](v) if k in self.ca_blocks else v
        return out

def build_faster_rcnn_ca(num_fg_classes: int = 7, imgsz: int = 640, pretrained_coco: bool = True):
    """
    Membangun Faster R-CNN-ResNet50-FPN + Coordinate Attention.
    Catatan: Faster R-CNN menggunakan class 0 sebagai __background__,
    sehingga total output head = num_fg_classes + 1.
    """
    weights = d_models.FasterRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained_coco else None
    model = d_models.fasterrcnn_resnet50_fpn(
        weights=weights,
        min_size=imgsz,
        max_size=imgsz,
        rpn_pre_nms_top_n_train=1000,
        rpn_post_nms_top_n_train=500,
        rpn_pre_nms_top_n_test=500,
        rpn_post_nms_top_n_test=300,
        box_batch_size_per_image=256,
    )
    # Pasang Coordinate Attention pada FPN tanpa merusak bobot COCO ResNet50+FPN
    model.backbone = BackboneFPNWithCA(model.backbone, reduction=16)

    # Ganti FastRCNNPredictor untuk (num_fg_classes + 1 background)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_fg_classes + 1)
    return model

print("[OK] Arsitektur Faster R-CNN-ResNet50-FPN + Coordinate Attention (CA) siap digunakan.")


### 3. Dataset Loader, Evaluator Metrik COCO/VOC & Inisialisasi Bobot Pretrained

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

class TeaLeafFasterRCNNDataset(Dataset):
    """Dataset pembaca format YOLO (.txt) ke format Faster R-CNN ([x1,y1,x2,y2], label + 1)."""
    def __init__(self, data_dir: str, split: str = "train", imgsz: int = 640, augment: bool = False):
        self.img_dir = os.path.join(data_dir, "images", split)
        self.lbl_dir = os.path.join(data_dir, "labels", split)
        self.imgsz = imgsz
        self.augment = augment
        self.files = sorted([
            f for f in os.listdir(self.img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx: int):
        fname = self.files[idx]
        img_path = os.path.join(self.img_dir, fname)
        lbl_path = os.path.join(self.lbl_dir, os.path.splitext(fname)[0] + ".txt")

        img = Image.open(img_path).convert("RGB")
        orig_w, orig_h = img.size

        boxes, labels = [], []
        if os.path.exists(lbl_path):
            with open(lbl_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cid = int(parts[0])
                        xc, yc, bw, bh = map(float, parts[1:5])
                        x1 = max(0.0, (xc - bw / 2.0) * orig_w)
                        y1 = max(0.0, (yc - bh / 2.0) * orig_h)
                        x2 = min(float(orig_w), (xc + bw / 2.0) * orig_w)
                        y2 = min(float(orig_h), (yc + bh / 2.0) * orig_h)
                        if x2 > x1 + 1.0 and y2 > y1 + 1.0:
                            boxes.append([x1, y1, x2, y2])
                            labels.append(cid + 1)  # +1 karena 0 adalah background

        boxes = torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64) if len(boxes) > 0 else torch.zeros((0,), dtype=torch.int64)

        if self.augment and random.random() < 0.5:
            img = TF.hflip(img)
            if len(boxes) > 0:
                nx1 = orig_w - boxes[:, 2]
                nx2 = orig_w - boxes[:, 0]
                boxes[:, 0], boxes[:, 2] = nx1, nx2

        sx, sy = self.imgsz / float(orig_w), self.imgsz / float(orig_h)
        img = TF.resize(img, [self.imgsz, self.imgsz])
        if len(boxes) > 0:
            boxes[:, [0, 2]] = (boxes[:, [0, 2]] * sx).clamp(0.0, float(self.imgsz))
            boxes[:, [1, 3]] = (boxes[:, [1, 3]] * sy).clamp(0.0, float(self.imgsz))
            valid = (boxes[:, 2] > boxes[:, 0] + 1.0) & (boxes[:, 3] > boxes[:, 1] + 1.0)
            boxes, labels = boxes[valid], labels[valid]

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
        }
        return TF.to_tensor(img), target

def detection_collate(batch):
    return tuple(zip(*batch))

def box_iou_np(boxes1: np.ndarray, boxes2: np.ndarray) -> np.ndarray:
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)
    a1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    a2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    lt = np.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = np.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = np.clip(rb - lt, 0.0, None)
    inter = wh[:, :, 0] * wh[:, :, 1]
    return inter / np.clip(a1[:, None] + a2[None, :] - inter, 1e-6, None)

def compute_ap_curve(tp: np.ndarray, fp: np.ndarray, n_gt: int):
    if n_gt == 0 or len(tp) == 0:
        return 0.0, np.array([0.0, 1.0]), np.array([0.0, 0.0])
    tp_c = np.cumsum(tp)
    fp_c = np.cumsum(fp)
    rec = tp_c / float(n_gt)
    prec = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([1.0], prec, [0.0]))
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
    return ap, mrec, mpre

@torch.no_grad()
def evaluate_faster_rcnn(model, loader, class_names, device=DEVICE, conf_thresh=0.25, save_plots_dir=None):
    """Evaluasi lengkap Precision, Recall, F1, mAP@0.50, mAP@0.50:0.95, Confusion Matrix & PR Curve."""
    model.eval()
    n_cls = len(class_names)
    preds_all, gts_all = [], []

    for images, targets in loader:
        images_gpu = [img.to(device) for img in images]
        outputs = model(images_gpu)
        for out, tgt in zip(outputs, targets):
            pb = out["boxes"].detach().cpu().numpy()
            ps = out["scores"].detach().cpu().numpy()
            pl = out["labels"].detach().cpu().numpy() - 1
            gb = tgt["boxes"].cpu().numpy()
            gl = tgt["labels"].cpu().numpy() - 1
            preds_all.append((pb, ps, pl))
            gts_all.append((gb, gl))

    iou_thresholds = np.linspace(0.50, 0.95, 10)
    per_class = []
    pr_curves = {}

    for c in range(n_cls):
        n_gt_c = sum(int(np.sum(gl == c)) for _, gl in gts_all)
        aps_iou = []
        for t_idx, iou_t in enumerate(iou_thresholds):
            scores_c, tp_c, fp_c = [], [], []
            for (pb, ps, pl), (gb, gl) in zip(preds_all, gts_all):
                mask_p = (pl == c) & (ps >= 0.01)
                pb_c, ps_c = pb[mask_p], ps[mask_p]
                gb_c = gb[gl == c]
                if len(pb_c) == 0:
                    continue
                order = np.argsort(-ps_c)
                pb_c, ps_c = pb_c[order], ps_c[order]
                ious = box_iou_np(pb_c, gb_c)
                matched = set()
                for i_p in range(len(pb_c)):
                    scores_c.append(float(ps_c[i_p]))
                    best_iou, best_g = 0.0, -1
                    for i_g in range(len(gb_c)):
                        if i_g not in matched and ious[i_p, i_g] > best_iou:
                            best_iou, best_g = float(ious[i_p, i_g]), i_g
                    if best_iou >= iou_t:
                        tp_c.append(1); fp_c.append(0); matched.add(best_g)
                    else:
                        tp_c.append(0); fp_c.append(1)
            if len(scores_c) > 0:
                ord_all = np.argsort(-np.array(scores_c))
                ap_val, mrec, mpre = compute_ap_curve(np.array(tp_c)[ord_all], np.array(fp_c)[ord_all], n_gt_c)
            else:
                ap_val, mrec, mpre = 0.0, np.array([0.0, 1.0]), np.array([0.0, 0.0])
            aps_iou.append(ap_val)
            if t_idx == 0:
                pr_curves[c] = (mrec, mpre, ap_val)

        # Hitung Precision & Recall pada conf_thresh (IoU >= 0.50)
        tp50, fp50, fn50 = 0, 0, 0
        for (pb, ps, pl), (gb, gl) in zip(preds_all, gts_all):
            mask_p = (pl == c) & (ps >= conf_thresh)
            pb_c, ps_c = pb[mask_p], ps[mask_p]
            gb_c = gb[gl == c]
            if len(pb_c) > 0:
                order = np.argsort(-ps_c)
                pb_c = pb_c[order]
            ious = box_iou_np(pb_c, gb_c)
            matched = set()
            for i_p in range(len(pb_c)):
                best_iou, best_g = 0.0, -1
                for i_g in range(len(gb_c)):
                    if i_g not in matched and ious[i_p, i_g] > best_iou:
                        best_iou, best_g = float(ious[i_p, i_g]), i_g
                if best_iou >= 0.50:
                    tp50 += 1; matched.add(best_g)
                else:
                    fp50 += 1
            fn50 += (len(gb_c) - len(matched))

        p_c = tp50 / (tp50 + fp50 + 1e-9) if (tp50 + fp50) > 0 else 0.0
        r_c = tp50 / (tp50 + fn50 + 1e-9) if (tp50 + fn50) > 0 else 0.0
        f1_c = 2 * p_c * r_c / (p_c + r_c + 1e-9)
        if n_gt_c > 0:
            per_class.append({
                "class_id": c,
                "class_name": class_names[c],
                "precision": float(p_c),
                "recall": float(r_c),
                "f1": float(f1_c),
                "ap50": float(aps_iou[0]),
                "ap50_95": float(np.mean(aps_iou)),
            })

    mp = float(np.mean([x["precision"] for x in per_class])) if per_class else 0.0
    mr = float(np.mean([x["recall"] for x in per_class])) if per_class else 0.0
    map50 = float(np.mean([x["ap50"] for x in per_class])) if per_class else 0.0
    map50_95 = float(np.mean([x["ap50_95"] for x in per_class])) if per_class else 0.0
    f1 = 2 * mp * mr / (mp + mr + 1e-9)

    # Simpan Confusion Matrix Normalized & BoxPR_curve jika diminta
    if save_plots_dir is not None:
        os.makedirs(save_plots_dir, exist_ok=True)
        cm = np.zeros((n_cls, n_cls), dtype=np.float32)
        for (pb, ps, pl), (gb, gl) in zip(preds_all, gts_all):
            keep = ps >= conf_thresh
            pb_k, ps_k, pl_k = pb[keep], ps[keep], pl[keep]
            ious = box_iou_np(gb, pb_k)
            for i_g, g_cls in enumerate(gl):
                if 0 <= g_cls < n_cls and len(pb_k) > 0:
                    best_p = int(np.argmax(ious[i_g]))
                    if ious[i_g, best_p] >= 0.50 and 0 <= pl_k[best_p] < n_cls:
                        cm[g_cls, pl_k[best_p]] += 1.0
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_norm = np.divide(cm, np.maximum(row_sums, 1e-9))

        fig, ax = plt.subplots(figsize=(9, 7.5))
        im = ax.imshow(cm_norm, cmap="Blues", vmin=0.0, vmax=1.0)
        plt.colorbar(im, ax=ax)
        ax.set_xticks(range(n_cls)); ax.set_xticklabels(class_names, rotation=40, ha="right")
        ax.set_yticks(range(n_cls)); ax.set_yticklabels(class_names)
        ax.set_xlabel("Predicted Class", fontweight="bold"); ax.set_ylabel("True Class", fontweight="bold")
        ax.set_title("Normalized Confusion Matrix - Faster R-CNN + CA", fontweight="bold")
        for r_i in range(n_cls):
            for c_i in range(n_cls):
                val = cm_norm[r_i, c_i]
                ax.text(c_i, r_i, f"{val:.2f}", ha="center", va="center", color="white" if val > 0.5 else "black", fontsize=9)
        plt.tight_layout()
        plt.savefig(os.path.join(save_plots_dir, "confusion_matrix_normalized.png"), dpi=180)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(9, 6.5))
        for c in range(n_cls):
            if c in pr_curves:
                mrec, mpre, ap_v = pr_curves[c]
                ax.plot(mrec, mpre, lw=2, label=f"{class_names[c]} ({ap_v:.3f})")
        ax.plot([0, 1], [map50, map50], "k--", lw=2, label=f"all classes mAP@0.5 = {map50:.3f}")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        ax.set_xlabel("Recall", fontweight="bold"); ax.set_ylabel("Precision", fontweight="bold")
        ax.set_title("Precision-Recall Curve - Faster R-CNN + CA", fontweight="bold")
        ax.grid(True, alpha=0.3); ax.legend(loc="lower left", fontsize=9)
        plt.tight_layout()
        plt.savefig(os.path.join(save_plots_dir, "BoxPR_curve.png"), dpi=180)
        plt.close(fig)

    return {
        "mp": mp, "mr": mr, "f1": f1, "map50": map50, "map50_95": map50_95,
        "per_class": per_class
    }

# Simpan checkpoint inisialisasi COCO + Coordinate Attention
REMAPPED_PT = "/content/faster_rcnn_ca_coco_init.pth"
_init_model = build_faster_rcnn_ca(num_fg_classes=len(CLASS_NAMES), imgsz=640, pretrained_coco=True)
tot_params = sum(p.numel() for p in _init_model.parameters())
ca_params = sum(p.numel() for p in _init_model.backbone.ca_blocks.parameters())
torch.save({"model_state_dict": _init_model.state_dict(), "class_names": CLASS_NAMES}, REMAPPED_PT)
del _init_model
print(f"[OK] Bobot awal COCO + CA disimpan ke: {REMAPPED_PT}")
print(f"     Total Parameter: {tot_params:,} (Termasuk 5x CoordAtt FPN: {ca_params:,} parameter)")


### 4. Pelatihan Model Faster R-CNN + Coordinate Attention
*(Menggunakan PyTorch AMP Mixed Precision agar pelatihan 50 Epoch berjalan cepat di GPU Colab. Atur `IS_FINETUNE = True` jika ingin Fine-Tuning)*

In [ ]:
import csv

def load_flexible_weights(model: nn.Module, ckpt_path: str):
    """Memuat state_dict secara aman meskipun jumlah kelas bertambah (misal +Ranting saat Fine-Tuning)."""
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    src_sd = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    dst_sd = model.state_dict()
    matched = 0
    for k, v in src_sd.items():
        if k in dst_sd:
            if dst_sd[k].shape == v.shape:
                dst_sd[k] = v
                matched += 1
            elif dst_sd[k].ndim == v.ndim and dst_sd[k].shape[1:] == v.shape[1:]:
                n_copy = min(dst_sd[k].shape[0], v.shape[0])
                dst_sd[k][:n_copy] = v[:n_copy]
                matched += 1
    model.load_state_dict(dst_sd, strict=True)
    return matched, len(dst_sd)

def run_faster_rcnn_training(is_finetune: bool = False):
    epochs = 25 if is_finetune else 50
    batch_size = 8
    imgsz = 640
    run_name = "faster_rcnn_ca_finetune" if is_finetune else "faster_rcnn_ca_50ep"
    project_runs = "/content/phase3_runs"
    run_dir = os.path.join(project_runs, run_name)
    weights_dir = os.path.join(run_dir, "weights")
    os.makedirs(weights_dir, exist_ok=True)

    model = build_faster_rcnn_ca(num_fg_classes=len(CLASS_NAMES), imgsz=imgsz, pretrained_coco=False)
    if is_finetune:
        s1_candidates = (
            glob.glob(f"{project_runs}/faster_rcnn_ca_50ep/weights/best.pth") +
            glob.glob("/content/drive/MyDrive/SKRIPSI/**/faster_rcnn_ca_50ep/**/best.pth", recursive=True)
        )
        init_w = s1_candidates[0] if s1_candidates else REMAPPED_PT
        print(f"[INFO] Mode Fine-Tuning aktif. Memuat bobot tahap 1: {init_w}")
        # Freeze layer awal ResNet (conv1 & layer1) agar fitur dasar stabil
        for name, p in model.backbone.body.named_parameters():
            if name.startswith(("conv1", "bn1", "layer1")):
                p.requires_grad = False
    else:
        init_w = REMAPPED_PT
        print(f"[INFO] Mode Training Dasar (50 Epoch) aktif. Memuat: {init_w}")

    m_cnt, tot_cnt = load_flexible_weights(model, init_w)
    print(f"[OK] {m_cnt}/{tot_cnt} tensor berhasil dimuat.")
    model.to(DEVICE)

    train_ds = TeaLeafFasterRCNNDataset(DATA_DIR, split="train", imgsz=imgsz, augment=True)
    val_ds = TeaLeafFasterRCNNDataset(DATA_DIR, split="val", imgsz=imgsz, augment=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=detection_collate, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, collate_fn=detection_collate, pin_memory=True)

    params = [p for p in model.parameters() if p.requires_grad]
    if is_finetune:
        optimizer = torch.optim.AdamW(params, lr=1.5e-4, weight_decay=5e-4)
    else:
        optimizer = torch.optim.SGD(params, lr=0.004, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

    csv_path = os.path.join(run_dir, "results.csv")
    best_pt = os.path.join(weights_dir, "best.pth")
    last_pt = os.path.join(weights_dir, "last.pth")
    best_map50 = -1.0
    no_improve = 0
    patience = 15

    with open(csv_path, "w", newline="", encoding="utf-8") as cf:
        writer = csv.writer(cf)
        writer.writerow(["epoch", "train/box_loss", "train/cls_loss", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"])

    print(f"[INFO] Memulai pelatihan {run_name} ({epochs} Epoch, Batch={batch_size}, {imgsz}x{imgsz})...")
    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        sum_box, sum_cls, n_steps = 0.0, 0.0, 0

        for step, (images, targets) in enumerate(train_loader, 1):
            images = [img.to(DEVICE, non_blocking=True) for img in images]
            targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]

            # Warmup ringan di Epoch 1
            if ep == 1 and not is_finetune and step <= min(200, len(train_loader)):
                warm_lr = 0.004 * (step / float(min(200, len(train_loader))))
                for pg in optimizer.param_groups:
                    pg["lr"] = max(1e-4, warm_lr)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                loss_dict = model(images, targets)
                b_loss = loss_dict["loss_box_reg"] + loss_dict["loss_rpn_box_reg"]
                c_loss = loss_dict["loss_classifier"] + loss_dict["loss_objectness"]
                total_loss = b_loss + c_loss

            if not math.isfinite(float(total_loss.item())):
                continue

            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, max_norm=10.0)
            scaler.step(optimizer)
            scaler.update()

            sum_box += float(b_loss.item())
            sum_cls += float(c_loss.item())
            n_steps += 1

        scheduler.step()
        avg_box = sum_box / max(1, n_steps)
        avg_cls = sum_cls / max(1, n_steps)

        val_res = evaluate_faster_rcnn(model, val_loader, CLASS_NAMES, device=DEVICE, conf_thresh=0.25)
        dt = time.time() - t0
        print(
            f"Epoch [{ep:02d}/{epochs:02d}] ({dt:.1f}s) | "
            f"BoxLoss: {avg_box:.4f} | ClsLoss: {avg_cls:.4f} | "
            f"P: {val_res['mp']:.4f} | R: {val_res['mr']:.4f} | "
            f"mAP50: {val_res['map50']:.4f} | mAP50-95: {val_res['map50_95']:.4f}"
        )

        with open(csv_path, "a", newline="", encoding="utf-8") as cf:
            csv.writer(cf).writerow([ep, f"{avg_box:.5f}", f"{avg_cls:.5f}", f"{val_res['mp']:.5f}", f"{val_res['mr']:.5f}", f"{val_res['map50']:.5f}", f"{val_res['map50_95']:.5f}"])

        ckpt_payload = {"epoch": ep, "model_state_dict": model.state_dict(), "class_names": CLASS_NAMES, "val_map50": val_res["map50"]}
        torch.save(ckpt_payload, last_pt)
        if val_res["map50"] >= best_map50:
            best_map50 = val_res["map50"]
            torch.save(ckpt_payload, best_pt)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"[EARLY STOPPING] Tidak ada peningkatan mAP50 selama {patience} epoch.")
                break

    print(f"[SUKSES] Pelatihan selesai! Bobot terbaik (Val mAP50={best_map50:.4f}) disimpan di: {best_pt}")
    return best_pt, run_name, project_runs

IS_FINETUNE = False  # Set True jika ingin langsung Fine-Tuning
BEST_WEIGHTS, RUN_NAME, PROJECT_RUNS = run_faster_rcnn_training(is_finetune=IS_FINETUNE)


### 5. Evaluasi Lengkap pada Held-Out Test Set (529 Citra, 640x640)

In [ ]:
import pandas as pd
from IPython.display import display

ckpt_eval = torch.load(BEST_WEIGHTS, map_location="cpu", weights_only=False)
eval_classes = ckpt_eval.get("class_names", CLASS_NAMES) if isinstance(ckpt_eval, dict) else CLASS_NAMES

eval_model = build_faster_rcnn_ca(num_fg_classes=len(eval_classes), imgsz=640, pretrained_coco=False)
load_flexible_weights(eval_model, BEST_WEIGHTS)
eval_model.to(DEVICE)

test_ds = TeaLeafFasterRCNNDataset(DATA_DIR, split="test", imgsz=640, augment=False)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2, collate_fn=detection_collate)

test_eval_dir = os.path.join(PROJECT_RUNS, RUN_NAME, "test_eval")
metrics = evaluate_faster_rcnn(eval_model, test_loader, eval_classes, device=DEVICE, conf_thresh=0.25, save_plots_dir=test_eval_dir)

rows = []
for item in metrics["per_class"]:
    rows.append({
        "Kelas": item["class_name"],
        "Precision": f"{item['precision']:.4f}",
        "Recall": f"{item['recall']:.4f}",
        "F1-Score": f"{item['f1']:.4f}",
        "mAP@0.50": f"{item['ap50']:.4f}",
        "mAP@0.50:0.95": f"{item['ap50_95']:.4f}",
    })

rows.append({
    "Kelas": "== MEAN / OVERALL ==",
    "Precision": f"{metrics['mp']:.4f}",
    "Recall": f"{metrics['mr']:.4f}",
    "F1-Score": f"{metrics['f1']:.4f}",
    "mAP@0.50": f"{metrics['map50']:.4f}",
    "mAP@0.50:0.95": f"{metrics['map50_95']:.4f}",
})

df = pd.DataFrame(rows)
print("\n" + "=" * 78)
print("  HASIL EVALUASI TEST SET (529 CITRA) - FASTER R-CNN + COORDINATE ATTENTION")
print("=" * 78)
display(df)


### 6. Visualisasi Kurva Konvergensi HD & Confusion Matrix

In [ ]:
from IPython.display import Image as IPyImage, display, Markdown

csv_path = os.path.join(PROJECT_RUNS, RUN_NAME, "results.csv")
if os.path.exists(csv_path):
    epochs_list, box_l, cls_l, prec_l, rec_l, m50_l, m95_l = [], [], [], [], [], [], []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for r in reader:
            clean = {k.strip(): v.strip() for k, v in r.items() if k}
            try:
                epochs_list.append(int(float(clean.get("epoch", len(epochs_list) + 1))))
                box_l.append(float(clean.get("train/box_loss", 0.0)))
                cls_l.append(float(clean.get("train/cls_loss", 0.0)))
                prec_l.append(float(clean.get("metrics/precision(B)", 0.0)))
                rec_l.append(float(clean.get("metrics/recall(B)", 0.0)))
                m50_l.append(float(clean.get("metrics/mAP50(B)", 0.0)))
                m95_l.append(float(clean.get("metrics/mAP50-95(B)", 0.0)))
            except ValueError:
                continue

    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    fig.suptitle(f"Kurva Konvergensi Pelatihan Faster R-CNN + CA ({len(epochs_list)} Epoch, 640x640)", fontsize=15, fontweight="bold")

    axes[0, 0].plot(epochs_list, box_l, color="#1f77b4", lw=2); axes[0, 0].set_title("Box Loss (RPN + RoI)", fontweight="bold"); axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].plot(epochs_list, cls_l, color="#d62728", lw=2); axes[0, 1].set_title("Classification Loss (RPN + RoI)", fontweight="bold"); axes[0, 1].grid(True, alpha=0.3)
    axes[0, 2].plot(epochs_list, prec_l, color="#2ca02c", lw=2, label="Precision"); axes[0, 2].plot(epochs_list, rec_l, color="#e377c2", lw=2, label="Recall"); axes[0, 2].legend(); axes[0, 2].set_title("Precision & Recall", fontweight="bold"); axes[0, 2].grid(True, alpha=0.3)
    axes[1, 0].plot(epochs_list, m50_l, color="#17becf", lw=2); axes[1, 0].set_title("mAP@0.50", fontweight="bold"); axes[1, 0].grid(True, alpha=0.3)
    axes[1, 1].plot(epochs_list, m95_l, color="#bcbd22", lw=2); axes[1, 1].set_title("mAP@0.50:0.95", fontweight="bold"); axes[1, 1].grid(True, alpha=0.3)
    f1_l = [2 * p * r / (p + r + 1e-9) for p, r in zip(prec_l, rec_l)]
    axes[1, 2].plot(epochs_list, f1_l, color="#8c564b", lw=2); axes[1, 2].set_title("F1-Score", fontweight="bold"); axes[1, 2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

display(Markdown("#### Confusion Matrix & PR Curve pada Test Set:"))
for p_img in [
    f"{PROJECT_RUNS}/{RUN_NAME}/test_eval/confusion_matrix_normalized.png",
    f"{PROJECT_RUNS}/{RUN_NAME}/test_eval/BoxPR_curve.png"
]:
    if os.path.exists(p_img):
        display(IPyImage(p_img, width=700))


### 7. Engine Deteksi Lapangan (v2.7): Multi-Daun + Background Blur + Twig/Fern Rejection + 2-View TTA (Faster R-CNN)

In [ ]:
import cv2

class TeaLeafFieldDetector:
    def __init__(self, weights_path: str):
        ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
        self.class_names = ckpt.get("class_names", CLASS_NAMES) if isinstance(ckpt, dict) else CLASS_NAMES
        self.class_colors = CLASS_COLORS
        self.device = DEVICE
        self.model = build_faster_rcnn_ca(num_fg_classes=len(self.class_names), imgsz=640, pretrained_coco=False)
        load_flexible_weights(self.model, weights_path)
        self.model.to(self.device)
        self.model.eval()

    def _get_leaf_mask(self, img_bgr):
        H, W = img_bgr.shape[:2]
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
        b, g, r = img_bgr[:, :, 0].astype(np.float32), img_bgr[:, :, 1].astype(np.float32), img_bgr[:, :, 2].astype(np.float32)
        exg = 2.0 * g - r - b

        is_woody = (h < 13) | (h > 150) | (r > g + 20.0) | ((r > g + 4.0) & (g - b < 32.0)) | (exg < 8.0)
        is_gray_twig = (s < 35) & (exg < 22.0)
        is_shadow = (v < 48)
        is_glossy_leaf = (h >= 18) & (h <= 88) & (s >= 20) & (v >= 85) & (g > r + 10.0) & (exg >= 14.0)

        is_leaf = (((h >= 13) & (h <= 90) & (s >= 35) & (v >= 48) & (exg >= 10.0) & (~is_gray_twig)) | is_glossy_leaf) & (~is_woody) & (~is_shadow)
        mask = (is_leaf.astype(np.uint8)) * 255

        k_close = max(7, int(min(H, W) * 0.013) | 1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close)))
        k_open = max(5, int(min(H, W) * 0.012) | 1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open)))
        return mask

    def _get_white_twig_mask(self, crop_bgr):
        hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
        s, v = hsv[:, :, 1].astype(np.float32), hsv[:, :, 2].astype(np.float32)
        b, g, r = crop_bgr[:, :, 0].astype(np.float32), crop_bgr[:, :, 1].astype(np.float32), crop_bgr[:, :, 2].astype(np.float32)
        exg = 2.0 * g - r - b
        is_white_twig = ((s < 62.0) & (v >= 82.0) & (np.abs(r - b) < 32.0) & (exg < 30.0)) | ((v >= 165.0) & (s < 88.0) & (r >= g - 12.0))
        m = (is_white_twig.astype(np.uint8)) * 255
        return cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))

    def _check_symptom(self, crop_bgr, cnt_local):
        ch, cw = crop_bgr.shape[:2]
        mask = np.zeros((ch, cw), dtype=np.uint8)
        cv2.drawContours(mask, [cnt_local], -1, 255, -1)
        erode_k = max(5, int(min(ch, cw) * 0.07) | 1)
        inner = cv2.erode(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_k, erode_k)))
        white_twig = self._get_white_twig_mask(crop_bgr)
        pure_lamina = cv2.bitwise_and(inner, cv2.bitwise_not(white_twig))
        if np.count_nonzero(pure_lamina) < 50:
            pure_lamina = cv2.bitwise_and(mask, cv2.bitwise_not(white_twig))
        if np.count_nonzero(pure_lamina) == 0:
            return False, 0.0

        hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
        h, s, v = hsv[:, :, 0][pure_lamina > 0], hsv[:, :, 1][pure_lamina > 0], hsv[:, :, 2][pure_lamina > 0]
        b, g, r = crop_bgr[:, :, 0][pure_lamina > 0], crop_bgr[:, :, 1][pure_lamina > 0], crop_bgr[:, :, 2][pure_lamina > 0]

        spots = (v < 76.0) | ((r > g - 1.0) & (r > b + 26.0) & (s >= 46.0) & (v < 162.0)) | ((h < 17.0) & (s >= 50.0) & (v < 155.0))
        bronze = (h >= 12.0) & (h <= 24.0) & (s >= 55.0) & (r >= 0.86 * g) & (r > b + 28.0) & (v < 165.0)
        spot_ratio = float(np.mean(spots))
        bronze_ratio = float(np.mean(bronze))
        is_diseased = (spot_ratio >= 0.032) or (bronze_ratio >= 0.18)
        return is_diseased, max(spot_ratio * 3.5, bronze_ratio * 1.8)

    def segment_leaves(self, img_bgr):
        H, W = img_bgr.shape[:2]
        img_area = float(H * W)
        base_mask = self._get_leaf_mask(img_bgr)

        bg_blur = cv2.medianBlur(img_bgr, max(15, int(min(H, W) * 0.032) | 1))
        step_img = cv2.bilateralFilter(bg_blur, 7, 35, 35)

        pad = max(12, int(min(H, W) * 0.025))
        padded = cv2.copyMakeBorder(base_mask, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
        dist = cv2.distanceTransform(padded, cv2.DIST_L2, 5)[pad:pad+H, pad:pad+W]
        dist_s = cv2.GaussianBlur(dist, (15, 15), 3.0)

        nms_rad = max(7, int(min(H, W) * 0.035))
        k_nms = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (nms_rad * 2 + 1, nms_rad * 2 + 1))
        local_max = (dist_s == cv2.dilate(dist_s, k_nms)) & (dist_s >= 6.5) & (base_mask > 0)

        peak_ys, peak_xs = np.where(local_max)
        peaks = sorted([(float(dist[py, px]), int(px), int(py)) for py, px in zip(peak_ys, peak_xs)], reverse=True)

        markers = np.zeros((H, W), dtype=np.int32)
        markers[base_mask == 0] = 1
        seeds = []
        for idx_s, (r_peak, px, py) in enumerate(peaks, start=2):
            if any(np.hypot(px - sx, py - sy) < max(10.0, 0.62 * min(r_peak, sr)) for _, sr, sx, sy in seeds):
                continue
            seeds.append((idx_s, r_peak, px, py))
            cv2.circle(markers, (px, py), max(4, int(0.30 * r_peak)), idx_s, -1)

        cv2.watershed(step_img, markers)
        candidates = []
        yy, xx = np.ogrid[:H, :W]

        for inst_id, r_peak, px, py in seeds:
            m_inst = ((markers == inst_id) & (base_mask > 0)).astype(np.uint8) * 255
            if np.count_nonzero(m_inst) < int(img_area * 0.0065):
                continue
            rad_lim = max(32.0, 2.35 * r_peak)
            reach = (((xx - px) / rad_lim) ** 2 + ((yy - py) / rad_lim) ** 2) <= 1.0
            m_inst[~reach] = 0

            cnts, _ = cv2.findContours(m_inst, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not cnts: continue
            c = max(cnts, key=cv2.contourArea)
            area = float(cv2.contourArea(c))
            perim = float(cv2.arcLength(c, True))
            circularity = (4.0 * np.pi * area) / (perim * perim + 1e-6)
            hull = cv2.convexHull(c)
            solidity = area / (float(cv2.contourArea(hull)) + 1e-6)
            bx, by, bw, bh = cv2.boundingRect(c)
            ar = max(bw, bh) / max(1.0, min(bw, bh))
            extent = area / float(max(1, bw * bh))
            box_green_ratio = float(np.mean(base_mask[by:by+bh, bx:bx+bw] > 0))

            # Filter ketat daun tunggal elips (tolak rumpun daun raksasa, pakis bergerigi, & tanah)
            if (area / img_area < 0.0065) or (area / img_area > 0.16) or (ar > 3.6) or (solidity < 0.68) or (circularity < 0.22) or (extent < 0.46) or (box_green_ratio < 0.45):
                continue

            crop = img_bgr[by:by+bh, bx:bx+bw]
            c_local = c.copy(); c_local[:, :, 0] -= bx; c_local[:, :, 1] -= by
            mask_c = np.zeros((bh, bw), dtype=np.uint8)
            cv2.drawContours(mask_c, [c_local], -1, 255, -1)
            inside = mask_c > 0

            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY).astype(np.float32)
            gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
            mag = np.hypot(gx, gy) / 4.0
            k_rng = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
            l_rng = cv2.dilate(gray, k_rng) - cv2.erode(gray, k_rng)

            ezones = inside & (l_rng >= 18.0)
            acutance = float(np.percentile(mag[ezones] / (l_rng[ezones] + 8.0), 80)) if np.count_nonzero(ezones) >= 20 else 0.25
            if acutance < 0.19:
                continue

            b_in, g_in, r_in = crop[:, :, 0][inside].astype(np.float32), crop[:, :, 1][inside].astype(np.float32), crop[:, :, 2][inside].astype(np.float32)
            exg_in = 2.0 * g_in - r_in - b_in
            twig_dark = float(np.mean((exg_in < 14.0) | (crop[:, :, 2][inside] < 78.0)))
            if solidity < 0.66 and twig_dark > 0.15:
                continue

            candidates.append((area * solidity, c, (bx, by, bx + bw, by + bh), solidity, m_inst))

        candidates.sort(key=lambda t: t[0], reverse=True)
        kept, kept_masks, kept_boxes = [], [], []
        for _, c, bbox, sol, m_cand in candidates:
            cand_px = float(np.count_nonzero(m_cand))
            bx1, by1, bx2, by2 = bbox
            b_area = float((bx2 - bx1) * (by2 - by1))
            overlap = False
            for m_prev, (px1, py1, px2, py2) in zip(kept_masks, kept_boxes):
                inter = float(np.count_nonzero((m_cand > 0) & (m_prev > 0)))
                if inter / max(cand_px, 1.0) > 0.22:
                    overlap = True; break
                ix1, iy1, ix2, iy2 = max(bx1, px1), max(by1, py1), min(bx2, px2), min(by2, py2)
                if ix2 > ix1 and iy2 > iy1:
                    if ((ix2 - ix1) * (iy2 - iy1)) / min(b_area, float((px2 - px1) * (py2 - py1))) > 0.45:
                        overlap = True; break
            if not overlap:
                kept.append((c, bbox, sol))
                kept_masks.append(m_cand)
                kept_boxes.append(bbox)
        return kept

    @torch.no_grad()
    def classify_leaf_tta(self, img_bgr, cnt, bbox):
        H, W = img_bgr.shape[:2]
        x1, y1, x2, y2 = bbox
        pad = max(6, int(max(x2 - x1, y2 - y1) * 0.04))
        cx1, cy1, cx2, cy2 = max(0, x1 - pad), max(0, y1 - pad), min(W, x2 + pad), min(H, y2 + pad)
        crop = img_bgr[cy1:cy2, cx1:cx2].copy()
        ch, cw = crop.shape[:2]

        m_crop = np.zeros((ch, cw), dtype=np.uint8)
        cnt_shifted = cnt.copy(); cnt_shifted[:, :, 0] -= cx1; cnt_shifted[:, :, 1] -= cy1
        cv2.drawContours(m_crop, [cnt_shifted], -1, 255, -1)

        w_twig = self._get_white_twig_mask(crop)
        clean_leaf = (m_crop > 0) & (w_twig == 0)
        med_bgr = np.median(crop[clean_leaf], axis=0).astype(np.uint8) if np.count_nonzero(clean_leaf) >= 40 else np.array([45, 145, 75], dtype=np.uint8)
        crop_healed = crop.copy()
        crop_healed[(m_crop > 0) & (w_twig > 0)] = med_bgr

        paper_color = np.array([236, 238, 238], dtype=np.uint8)
        side = int(max(ch, cw) * 1.36)
        canvas = np.full((side, side, 3), paper_color, dtype=np.uint8)
        y_off, x_off = (side - ch) // 2, (side - cw) // 2
        iso = np.full_like(crop_healed, paper_color)
        iso[m_crop > 0] = crop_healed[m_crop > 0]
        canvas[y_off:y_off+ch, x_off:x_off+cw] = iso

        has_symptom, _ = self._check_symptom(crop, cnt_shifted)
        canvas_640 = cv2.resize(canvas, (640, 640))
        flip_640 = cv2.flip(canvas_640, 1)
        t1 = TF.to_tensor(cv2.cvtColor(canvas_640, cv2.COLOR_BGR2RGB)).to(self.device)
        t2 = TF.to_tensor(cv2.cvtColor(flip_640, cv2.COLOR_BGR2RGB)).to(self.device)
        preds = self.model([t1, t2])

        scores = {c: [] for c in range(len(self.class_names))}
        for p in preds:
            lbls = p["labels"].detach().cpu().numpy() - 1
            confs = p["scores"].detach().cpu().numpy()
            for c_id, c_conf in zip(lbls, confs):
                if c_conf >= 0.05 and 0 <= int(c_id) < len(self.class_names):
                    scores[int(c_id)].append(float(c_conf))

        agg = {}
        for c_id, s_list in scores.items():
            if len(s_list) >= 2:
                agg[c_id] = min(0.99, float(np.mean(s_list[:2])) * 1.04)
            elif len(s_list) == 1:
                agg[c_id] = float(s_list[0]) * 0.95

        if not has_symptom:
            return (6, max(agg.get(6, 0.84), 0.84), self.class_names[6])

        dis_scores = {k: v for k, v in agg.items() if k != 6}
        if dis_scores:
            best_c = max(dis_scores, key=lambda k: dis_scores[k])
            return (best_c, dis_scores[best_c], self.class_names[best_c])
        return (1, 0.65, self.class_names[1])

    def blur_background(self, img_bgr, leaf_instances=None):
        if leaf_instances is None:
            leaf_instances = self.segment_leaves(img_bgr)
        H, W = img_bgr.shape[:2]
        fg_mask = np.zeros((H, W), dtype=np.uint8)
        for cnt, _, _ in leaf_instances:
            cv2.drawContours(fg_mask, [cnt], -1, 255, -1)
        alpha = (cv2.GaussianBlur(fg_mask, (21, 21), 0).astype(np.float32) / 255.0)[:, :, None]
        bg_blur = cv2.GaussianBlur(img_bgr, (55, 55), 0)
        return (img_bgr * alpha + bg_blur * (1.0 - alpha)).astype(np.uint8)

    def detect_and_visualize(self, img_bgr):
        H, W = img_bgr.shape[:2]
        leaf_instances = self.segment_leaves(img_bgr)
        img_bokeh = self.blur_background(img_bgr, leaf_instances)
        img_bbox = img_bokeh.copy()
        img_contour = img_bokeh.copy()
        overlay = img_contour.copy()

        n_diseased, n_healthy = 0, 0
        for cnt, bbox, sol in leaf_instances:
            cid, conf, cname = self.classify_leaf_tta(img_bgr, cnt, bbox)
            if cid == 7: continue
            color = self.class_colors[cid % len(self.class_colors)]
            if cid == 6: n_healthy += 1
            else: n_diseased += 1

            x1, y1, x2, y2 = bbox
            cv2.rectangle(img_bbox, (x1, y1), (x2, y2), color, 3)
            lbl = f"{cname} {conf:.2f}"
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 1)
            cv2.rectangle(img_bbox, (x1, max(th + 6, y1) - th - 6), (min(W - 1, x1 + tw + 6), max(th + 6, y1)), color, -1)
            cv2.putText(img_bbox, lbl, (x1 + 3, max(th + 6, y1) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)

            cv2.fillPoly(overlay, [cnt], color)
            cv2.drawContours(img_contour, [cnt], -1, color, 3)
            cv2.putText(img_contour, lbl, (max(4, int(cnt[:, 0, 0].min())), max(18, int(cnt[:, 0, 1].min()))), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)

        cv2.addWeighted(overlay, 0.24, img_contour, 0.76, 0, img_contour)

        clean_mask = self._get_leaf_mask(img_bgr)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        canny_leaf = cv2.bitwise_and(cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 30, 100), clean_mask)

        fig, axes = plt.subplots(2, 2, figsize=(16, 14))
        axes[0, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0, 0].set_title("1. Gambar Asli (Input Eksternal)", fontsize=13, fontweight="bold"); axes[0, 0].axis("off")
        axes[0, 1].imshow(cv2.cvtColor(img_bbox, cv2.COLOR_BGR2RGB)); axes[0, 1].set_title(f"2. Faster R-CNN + CA ({len(leaf_instances)} Daun: {n_diseased} Sakit, {n_healthy} Sehat)", fontsize=13, fontweight="bold"); axes[0, 1].axis("off")
        axes[1, 0].imshow(cv2.cvtColor(img_contour, cv2.COLOR_BGR2RGB)); axes[1, 0].set_title("3. Edge Contour Daun Fokus (Bokeh)", fontsize=13, fontweight="bold"); axes[1, 0].axis("off")
        axes[1, 1].imshow(canny_leaf, cmap="gray"); axes[1, 1].set_title("4. Canny Edge Map (Ranting & Blur Dieliminasi)", fontsize=13, fontweight="bold"); axes[1, 1].axis("off")
        plt.tight_layout()
        plt.show()

detector = TeaLeafFieldDetector(BEST_WEIGHTS)
print("[OK] TeaLeafFieldDetector berbasis Faster R-CNN + CA siap digunakan!")


### 8. Uji Coba Interaktif Gambar Kebun Sendiri (Upload Gambar)

In [ ]:
from google.colab import files

detector = TeaLeafFieldDetector(BEST_WEIGHTS)
print("Silakan pilih gambar daun teh dari perangkat Anda (.jpg / .png):")
uploaded = files.upload()

for fname in uploaded.keys():
    print(f"\nMemproses gambar: {fname} ...")
    img_bgr = cv2.imread(fname)
    if img_bgr is not None:
        if max(img_bgr.shape[:2]) > 1280:
            sc = 1280.0 / max(img_bgr.shape[:2])
            img_bgr = cv2.resize(img_bgr, (int(img_bgr.shape[1] * sc), int(img_bgr.shape[0] * sc)))
        detector.detect_and_visualize(img_bgr)
    else:
        print(f"[ERROR] Gagal membaca gambar: {fname}")


### 9. Backup Otomatis Hasil & Bobot Terbaik ke Google Drive

In [ ]:
BACKUP_DIR = "/content/drive/MyDrive/SKRIPSI/phase3_runs_backup"
os.makedirs(BACKUP_DIR, exist_ok=True)

target_run = os.path.join(PROJECT_RUNS, RUN_NAME)
if os.path.exists(target_run):
    dest_dir = os.path.join(BACKUP_DIR, RUN_NAME)
    shutil.copytree(target_run, dest_dir, dirs_exist_ok=True)
    print(f"[SUKSES] Folder hasil {RUN_NAME} berhasil dicadangkan ke Google Drive:")
    print(f"         {dest_dir}")
    !ls -lh "{dest_dir}/weights"
else:
    print(f"[WARNING] Folder {target_run} belum ditemukan.")


### 10. Pra-Label Otomatis (Auto-Prelabeling) Data Lapangan untuk Roboflow
Menggunakan langsung objek `detector` (`TeaLeafFieldDetector` Faster R-CNN + CA) beserta *Background Blur* untuk membuat pra-label data lapangan.

In [ ]:
FIELD_DIR = "/content/drive/MyDrive/SKRIPSI/data_lapangan"
OUT_DIR   = "/content/roboflow_prelabeled_frcnn"
shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_DIR, exist_ok=True)

with open(f"{OUT_DIR}/classes.txt", "w") as f:
    f.write("\n".join(CLASS_NAMES[:7] + ["Ranting"]) + "\n")

img_files = sorted(glob.glob(f"{FIELD_DIR}/*.*"))
for idx, img_path in enumerate(img_files, 1):
    img = cv2.imread(img_path)
    if img is None: continue
    if max(img.shape[:2]) > 1280:
        sc = 1280.0 / max(img.shape[:2])
        img = cv2.resize(img, (int(img.shape[1] * sc), int(img.shape[0] * sc)))
    H, W = img.shape[:2]
    leaves = detector.segment_leaves(img)
    img_blur = detector.blur_background(img, leaves)
    cv2.imwrite(f"{OUT_DIR}/lapangan_{idx:03d}.jpg", img_blur)

    lines = []
    for cnt, (x1, y1, x2, y2), _ in leaves:
        cls_id, _, _ = detector.classify_leaf_tta(img, cnt, (x1, y1, x2, y2))
        lines.append(f"{cls_id} {((x1+x2)/2)/W:.6f} {((y1+y2)/2)/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")

    with open(f"{OUT_DIR}/lapangan_{idx:03d}.txt", "w") as f:
        f.write("\n".join(lines))

zip_out = shutil.make_archive("/content/drive/MyDrive/SKRIPSI/roboflow_prelabeled_lapangan_frcnn", "zip", OUT_DIR)
print(f"[SUKSES] {len(img_files)} foto (Background Blur) selesai di-prelabel -> {zip_out}")


### 11. Gabungkan Data Lapangan Berlabel ke Dataset Utama & Fine-Tuning Faster R-CNN + CA
Memasukkan data lapangan berlabel (dari Roboflow `roboflow_lapangan.zip` atau hasil Pra-Label) ke dataset utama, lalu menjalankan Fine-Tuning.

In [ ]:
RF_ZIP = "/content/drive/MyDrive/SKRIPSI/roboflow_lapangan.zip"
PRE_ZIP_FRCNN = "/content/drive/MyDrive/SKRIPSI/roboflow_prelabeled_lapangan_frcnn.zip"
PRE_ZIP_YOLO = "/content/drive/MyDrive/SKRIPSI/roboflow_prelabeled_lapangan.zip"

if os.path.exists(RF_ZIP):
    shutil.unpack_archive(RF_ZIP, "/content/rf_data")
    for rf_sp, dst_sp in [("train", "train"), ("valid", "val"), ("test", "test")]:
        for kind in ["images", "labels"]:
            for p in glob.glob(f"/content/rf_data/{rf_sp}/{kind}/*.*"):
                shutil.copy2(p, f"{DATA_DIR}/{kind}/{dst_sp}/rf_{os.path.basename(p)}")
    CLASS_NAMES = CLASS_NAMES[:7] + ["Ranting"]
    print("[SUKSES] Data lapangan hasil koreksi Roboflow berhasil digabung!")
else:
    src_pre = "/content/roboflow_prelabeled_frcnn"
    if not os.path.exists(src_pre):
        for zp in [PRE_ZIP_FRCNN, PRE_ZIP_YOLO]:
            if os.path.exists(zp):
                shutil.unpack_archive(zp, src_pre)
                break
    imgs = sorted(glob.glob(f"{src_pre}/*.jpg"))
    for i, img_p in enumerate(imgs):
        txt_p = img_p.replace(".jpg", ".txt")
        sp = "val" if i % 5 == 0 else "train"
        shutil.copy2(img_p, f"{DATA_DIR}/images/{sp}/{os.path.basename(img_p)}")
        if os.path.exists(txt_p):
            shutil.copy2(txt_p, f"{DATA_DIR}/labels/{sp}/{os.path.basename(txt_p)}")
    print(f"[SUKSES] {len(imgs)} foto lapangan (Background Blur) berhasil dimasukkan ke dataset!")

# Jalankan Fine-Tuning Faster R-CNN + CA
FT_BEST_WEIGHTS, RUN_NAME, PROJECT_RUNS = run_faster_rcnn_training(is_finetune=True)
BEST_WEIGHTS = FT_BEST_WEIGHTS


### 12. Evaluasi Tuning (Metrik + Perbandingan Data Berlabel vs Hasil Prediksi Fine-Tuning)

In [ ]:
# 1. Evaluasi Metrik setelah Fine-Tuning (Validation Set)
w_pt = FT_BEST_WEIGHTS if "FT_BEST_WEIGHTS" in globals() else BEST_WEIGHTS
ckpt_ft = torch.load(w_pt, map_location="cpu", weights_only=False)
ft_classes = ckpt_ft.get("class_names", CLASS_NAMES) if isinstance(ckpt_ft, dict) else CLASS_NAMES

ft_model = build_faster_rcnn_ca(num_fg_classes=len(ft_classes), imgsz=640, pretrained_coco=False)
load_flexible_weights(ft_model, w_pt)
ft_model.to(DEVICE)

val_ds_ft = TeaLeafFasterRCNNDataset(DATA_DIR, split="val", imgsz=640, augment=False)
val_loader_ft = DataLoader(val_ds_ft, batch_size=8, shuffle=False, num_workers=2, collate_fn=detection_collate)
ft_eval = evaluate_faster_rcnn(ft_model, val_loader_ft, ft_classes, device=DEVICE, conf_thresh=0.25)
print(
    f"[HASIL FINE-TUNING FASTER R-CNN + CA] "
    f"Precision: {ft_eval['mp']:.4f} | Recall: {ft_eval['mr']:.4f} | "
    f"F1: {ft_eval['f1']:.4f} | mAP@0.50: {ft_eval['map50']:.4f} | mAP@0.50:0.95: {ft_eval['map50_95']:.4f}"
)

# 2. Tampilkan Perbandingan: KIRI (Data Lapangan Berlabel) vs KANAN (Hasil Prediksi Fine-Tuning Faster R-CNN + CA)
PRE_DIR = "/content/roboflow_prelabeled_frcnn"
if not os.path.exists(PRE_DIR):
    for zp in ["/content/drive/MyDrive/SKRIPSI/roboflow_prelabeled_lapangan_frcnn.zip", "/content/drive/MyDrive/SKRIPSI/roboflow_prelabeled_lapangan.zip"]:
        if os.path.exists(zp):
            shutil.unpack_archive(zp, PRE_DIR)
            break

detector_ft = TeaLeafFieldDetector(w_pt)
sample_imgs = sorted(glob.glob(f"{PRE_DIR}/*.jpg"))[:6]

for img_p in sample_imgs:
    img = cv2.imread(img_p)
    if img is None: continue
    H, W = img.shape[:2]
    img_gt = img.copy()
    txt_p = img_p.replace(".jpg", ".txt")

    n_gt = 0
    if os.path.exists(txt_p):
        for line in open(txt_p).read().splitlines():
            parts = line.split()
            if len(parts) != 5: continue
            cid = int(parts[0])
            xc, yc, bw, bh = [float(x) for x in parts[1:]]
            x1, y1, x2, y2 = int((xc - bw/2)*W), int((yc - bh/2)*H), int((xc + bw/2)*W), int((yc + bh/2)*H)
            col = CLASS_COLORS[cid % len(CLASS_COLORS)]
            cv2.rectangle(img_gt, (x1, y1), (x2, y2), col, 3)
            lbl_name = ft_classes[cid] if cid < len(ft_classes) else str(cid)
            cv2.putText(img_gt, lbl_name, (x1, max(20, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
            n_gt += 1

    leaves = detector_ft.segment_leaves(img)
    img_pred = detector_ft.blur_background(img, leaves)
    n_pred = 0
    for cnt, (x1, y1, x2, y2), _ in leaves:
        cid, conf, cname = detector_ft.classify_leaf_tta(img, cnt, (x1, y1, x2, y2))
        if cid == 7: continue
        col = CLASS_COLORS[cid % len(CLASS_COLORS)]
        cv2.rectangle(img_pred, (x1, y1), (x2, y2), col, 3)
        cv2.putText(img_pred, f"{cname} {conf:.2f}", (x1, max(20, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
        n_pred += 1

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB)); axes[0].set_title(f"Data Berlabel: {os.path.basename(img_p)} ({n_gt} Daun)", fontweight="bold"); axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB)); axes[1].set_title(f"Hasil Prediksi Faster R-CNN + CA ({n_pred} Daun)", fontweight="bold"); axes[1].axis("off")
    plt.tight_layout(); plt.show()
